# Zenodo Disaster Data Preprocessing

## Importing Packages

In [45]:
import os
import pandas as pd

## Reading Data

In [46]:
labels = {
    "no_disaster": 0,
    "earthquake": 1,
    "flood": 2,
    "hurricane": 3,
    "tornado": 4,
    "wildfire": 5
}

dfs = []

for file_name in [f for f in os.listdir("data/") if f.endswith(".ndjson")]:
    file_path = os.path.join("data/", file_name)
    label = labels[file_name.split("-")[0]]
    df = pd.read_json(file_path, lines=True)
    df["relevance"] = df["relevance"].replace(1, label)
    dfs.append(df)

In [47]:
combined_df = pd.concat(dfs, ignore_index=True)

In [48]:
combined_df.drop(columns=["id"], inplace=True)

In [49]:
combined_df.rename(columns={"relevance": "label"}, inplace=True)

In [50]:
combined_df.head()

,text,label
0,"earthquake in iloilo, philippines! my head's a...",1
1,new: felt <HASHTAG> earthquake <NUMBER> - boho...,1
2,"<NUMBER> earthquake, <NUMBER> m s of nueva vid...",1
3,just in: magnitude <NUMBER> earthquake: <NUMBE...,1
4,<NUMBER> earthquake recorded in the philippine...,1


In [51]:
combined_df.tail()

,text,label
128679,will it be easy? nope. will it be worth it? ab...,0
128680,<USER> all is cool here thank god!,0
128681,<USER> she's not adding up. she said she was <...,0
128682,"just listening to dave days' covers/originals,...",0
128683,"""anyone you're hoping to work with?"" demi: ""my...",0


In [52]:
combined_df.describe()

,label
count,128684.000000
mean,1.228327
std,1.460113
min,0.000000
25%,0.000000
50%,0.500000
75%,3.000000
max,5.000000


In [53]:
combined_df.isnull().sum()

text     0
label    0
dtype: int64

In [54]:
combined_df.nunique()

text     125019
label         6
dtype: int64

In [55]:
combined_df.dtypes

text     object
label     int64
dtype: object

In [56]:
df = combined_df

## Filtering Disaster Tweets

In [57]:
df = df[df["label"] != 0]

In [58]:
df.describe()

,label
count,64342.000000
mean,2.456654
std,1.116367
min,1.000000
25%,1.000000
50%,3.000000
75%,3.000000
max,5.000000


In [59]:
df.nunique()

text     62784
label        5
dtype: int64

## Removing Tags

In [60]:
tags = set(df["text"].str.findall(r"<\S+>").sum())
tags

{'</a>', '<HASHTAG>', '<NUMBER>', '<REPEAT>', '<SMILE>', '<URL>', '<USER>'}

In [61]:
tags.remove("</a>")

In [62]:
tags

{'<HASHTAG>', '<NUMBER>', '<REPEAT>', '<SMILE>', '<URL>', '<USER>'}

In [63]:
for tag in tags:
    df.loc[:, "text"] = df["text"].str.replace(tag, "")

In [64]:
set(df["text"].str.findall(r"<\S+>").sum())

{'</a>'}

## Removing Links

In [65]:
tco_links = set(df["text"].str.findall(r"\S*t\.co\S*").sum())
len(tco_links)

1363

In [66]:
http_links = set(df["text"].str.findall(r"\S*http\S*").sum())
len(http_links)

1415

In [67]:
links = tco_links | http_links
len(links)

1471

In [68]:
for link in links:
    df.loc[:, "text"] = df["text"].str.replace(link, "")

In [69]:
set(df["text"].str.findall(r"\S*t\.co\S*").sum())

set()

In [70]:
set(df["text"].str.findall(r"\S*http\S*").sum())

set()

## Removing Invalid Whitespace

In [71]:
df.loc[:, "text"] = df["text"].str.replace(r"\s+", " ", regex=True).str.strip()

In [72]:
if df["text"].str.contains(r"\s\s").any():
    print("Error: Consecutive whitespace.")
if df["text"].str.count(r"\s").sum() != df["text"].str.count(" ").sum():
    print("Error: Non-space whitespace.")

## Writing Data

In [73]:
df.to_csv("data/preprocessed_data.csv", sep="\t", encoding="utf-16", index=False)